# 00 - Environment

This notebook is the Jupyter front end of the same scan the terminal runs:

```bash
python -m apexgpt env
```

It reports the machine spec, every requirement from `requirements*.txt`, and
the settings ApexGPT resolves for this host - then applies them, so the
training and inference cells below use the same device, thread count and
precision policy as the command line.

In [ ]:
import sys
from pathlib import Path

# Works whether ApexGPT is pip-installed or merely cloned next to this notebook.
def _repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "apexgpt" / "__init__.py").exists():
            return candidate
    return start

ROOT = _repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import apexgpt
print("apexgpt", apexgpt.__version__, "from", Path(apexgpt.__file__).parent)

In [ ]:
from apexgpt.core.environment import bootstrap

report, settings = bootstrap()
print(report.to_text())

## Machine spec as a table

In [ ]:
print(report.to_markdown())

## Resolved settings

In [ ]:
print("device       :", settings.device)
print("backend      :", settings.backend)
print("threads      :", settings.threads)
print("amp          :", settings.amp)
print("checkpointing:", settings.checkpointing)
print("preset       :", settings.preset, "->", report.accelerator.precision)

## Re-scan for a specific backend

Detection is automatic, but you can ask what the settings *would* be for
another device. Asking for a backend that is absent is an error, never a
silent fallback to the CPU.

In [ ]:
from apexgpt.core.environment import scan

for candidate in ("cpu", "cuda", "mps", "xpu", "dml"):
    try:
        print(f"{candidate:>5}: {scan(device=candidate).settings.device}")
    except RuntimeError as exc:
        print(f"{candidate:>5}: unavailable ({str(exc)[:60]}...)")

## Requirements

The scan reads the requirements files, so it cannot drift from what
`python -m apexgpt setup --install` actually installs.

In [ ]:
for req in report.requirements:
    print(f"{req.status:>8}  {req.name:<20} {req.specifier}")

print()
print("missing:", [r.name for r in report.missing] or "none")

## Notes from this host

On a machine without an NVIDIA GPU the scan says so explicitly instead of
pretending CUDA exists.

In [ ]:
for note in report.notes:
    print("-", note)